<a href="https://colab.research.google.com/github/kamalahmeddlg/NSFW-Image-Detection-System/blob/main/nsfw_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import random
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/nsfw_recognition")

RAW_DIR = PROJECT_DIR / "1_data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "1_data" / "processed"
SPLITS_DIR = PROJECT_DIR / "1_data" / "splits"

TRAIN_DIR = SPLITS_DIR / "train"
VAL_DIR = SPLITS_DIR / "val"
TEST_DIR = SPLITS_DIR / "test"

MODELS_DIR = PROJECT_DIR / "3_models"
RESULTS_DIR = PROJECT_DIR / "4_results"
PLOTS_DIR = RESULTS_DIR / "plots"
REPORTS_DIR = RESULTS_DIR / "reports"
GRADCAM_DIR = RESULTS_DIR / "gradcam_outputs"

LOGS_DIR = PROJECT_DIR / "6_logs"

CLASS_NAMES = ["safe", "nsfw"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

for p in [
    RAW_DIR, PROCESSED_DIR, SPLITS_DIR,
    TRAIN_DIR / "safe", TRAIN_DIR / "nsfw",
    VAL_DIR / "safe", VAL_DIR / "nsfw",
    TEST_DIR / "safe", TEST_DIR / "nsfw",
    MODELS_DIR, PLOTS_DIR, REPORTS_DIR,
    GRADCAM_DIR / "safe", GRADCAM_DIR / "nsfw",
    LOGS_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Project folders ready")
print(PROJECT_DIR)

✅ Project folders ready
/content/drive/MyDrive/nsfw_recognition


In [ ]:
!pip install -q tensorflow opencv-python pillow scikit-learn matplotlib pandas

In [ ]:
from PIL import Image

In [ ]:
def get_image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in IMG_EXT]

def count_images(folder):
    return len(get_image_files(folder))

def remove_corrupt_images(folder):
    folder = Path(folder)
    removed = 0
    for f in get_image_files(folder):
        try:
            with Image.open(f) as img:
                img.verify()
            with Image.open(f) as img:
                img.load()
        except Exception as e:
            print(f"❌ Removing: {f.name} | {e}")
            try:
                f.unlink()
                removed += 1
            except:
                pass
    return removed

In [ ]:
safe_before = count_images(RAW_DIR / "safe")
nsfw_before = count_images(RAW_DIR / "nsfw")

print("Before cleaning")
print("Safe:", safe_before)
print("NSFW:", nsfw_before)

Before cleaning
Safe: 9529
NSFW: 0


In [ ]:
removed_safe = remove_corrupt_images(RAW_DIR / "safe")
removed_nsfw = remove_corrupt_images(RAW_DIR / "nsfw")

print("Removed Safe:", removed_safe)
print("Removed NSFW:", removed_nsfw)

Removed Safe: 0
Removed NSFW: 0


In [ ]:
safe_after = count_images(RAW_DIR / "safe")
nsfw_after = count_images(RAW_DIR / "nsfw")

summary = {
    "safe_after_cleaning": safe_after,
    "nsfw_after_cleaning": nsfw_after
}

with open(REPORTS_DIR / "data_check_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Data check completed")
print(summary)

✅ Data check completed
{'safe_after_cleaning': 9529, 'nsfw_after_cleaning': 18863}


In [ ]:
def clear_split_dirs():
    for split_root in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
        for cls in CLASS_NAMES:
            cls_dir = split_root / cls
            cls_dir.mkdir(parents=True, exist_ok=True)
            for file in cls_dir.iterdir():
                if file.is_file():
                    file.unlink()
                elif file.is_dir():
                    shutil.rmtree(file, ignore_errors=True)

In [ ]:
def split_and_copy():
    random.seed(SEED)
    report = {}

    clear_split_dirs()

    for cls in CLASS_NAMES:
        src = RAW_DIR / cls
        files = get_image_files(src)
        random.shuffle(files)

        total = len(files)
        train_n = int(total * 0.8)
        val_n = int(total * 0.1)
        test_n = total - train_n - val_n

        train_files = files[:train_n]
        val_files = files[train_n:train_n + val_n]
        test_files = files[train_n + val_n:]

        for f in train_files:
            shutil.copy2(f, TRAIN_DIR / cls / f.name)
        for f in val_files:
            shutil.copy2(f, VAL_DIR / cls / f.name)
        for f in test_files:
            shutil.copy2(f, TEST_DIR / cls / f.name)

        report[cls] = {
            "total": total,
            "train": len(train_files),
            "val": len(val_files),
            "test": len(test_files)
        }

    return report

In [ ]:
split_report = split_and_copy()

with open(REPORTS_DIR / "dataset_split_summary.json", "w") as f:
    json.dump(split_report, f, indent=2)

print("✅ Dataset split done")
print(json.dumps(split_report, indent=2))

✅ Dataset split done
{
  "safe": {
    "total": 9529,
    "train": 7623,
    "val": 952,
    "test": 954
  },
  "nsfw": {
    "total": 18863,
    "train": 15090,
    "val": 1886,
    "test": 1887
  }
}


In [ ]:
import tensorflow as tf
import numpy as np

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

Found 22713 files belonging to 2 classes.
Found 2838 files belonging to 2 classes.


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")

In [ ]:
baseline_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Rescaling(1./255),
    data_augmentation,
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

baseline_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,377 (364.75 KB)

 Trainable params: 93,377 (364.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "best_baseline.keras"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

In [ ]:
history_baseline = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

baseline_model.save(MODELS_DIR / "last_baseline.keras")
print("✅ Baseline training completed")

Epoch 1/10
684/710 ━━━━━━━━━━━━━━━━━━━━ 1:22 3s/step - accuracy: 0.7558 - auc: 0.8055 - loss: 0.4965

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

In [ ]:
inputs = tf.keras.layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / "best_model.keras"),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        verbose=1
    )
]

In [ ]:
history_transfer = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

model.save(MODELS_DIR / "last_model.keras")
print("✅ Transfer learning completed")

In [ ]:
model = tf.keras.models.load_model(MODELS_DIR / "best_model.keras")

In [ ]:
base_model = None
for layer in model.layers:
    if "efficientnetb0" in layer.name.lower():
        base_model = layer
        break

print(base_model)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

In [ ]:
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)

model.save(MODELS_DIR / "best_finetuned_model.keras")
print("✅ Fine-tuning completed")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import pandas as pd

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
).prefetch(tf.data.AUTOTUNE)

In [ ]:
model = tf.keras.models.load_model(MODELS_DIR / "best_finetuned_model.keras")

In [ ]:
y_true = np.concatenate([y.numpy().ravel() for _, y in test_ds])
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
print(report)

with open(REPORTS_DIR / "classification_report.txt", "w") as f:
    f.write(report)

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig(PLOTS_DIR / "roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], CLASS_NAMES)
plt.yticks([0, 1], CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.savefig(PLOTS_DIR / "confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
metrics_df = pd.DataFrame({
    "y_true": y_true,
    "y_prob": y_prob,
    "y_pred": y_pred
})
metrics_df.to_csv(REPORTS_DIR / "metrics.csv", index=False)

summary_text = f"Test AUC: {roc_auc:.4f}\nTotal Samples: {len(y_true)}\n"
with open(REPORTS_DIR / "evaluation_summary.txt", "w") as f:
    f.write(summary_text)

print("✅ Evaluation completed")

In [ ]:
import cv2
from tensorflow.keras.preprocessing import image

In [ ]:
model = tf.keras.models.load_model(MODELS_DIR / "best_finetuned_model.keras")

In [ ]:
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    return arr

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

In [ ]:
for layer in model.layers:
    print(layer.name)

In [ ]:
def make_gradcam_heatmap(img_array, model, base_model_name, last_conv_layer_name):
    base_model = model.get_layer(base_model_name)

    last_conv_layer = base_model.get_layer(last_conv_layer_name)

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [last_conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing import image
from pathlib import Path

In [ ]:
model = tf.keras.models.load_model(MODELS_DIR / "best_finetuned_model.keras")
model.summary()

In [ ]:
def get_img_array(img_path, size):
    img = image.load_img(img_path, target_size=size)
    arr = image.img_to_array(img).astype("float32")
    arr = np.expand_dims(arr, axis=0)
    return arr

In [ ]:
# nested backbone nikalne ke liye
def find_backbone(model, backbone_keywords=("efficientnet", "resnet", "mobilenet", "xception")):
    for layer in model.layers:
        name = layer.name.lower()
        if any(k in name for k in backbone_keywords):
            return layer
    return None

backbone = find_backbone(model)
print("Backbone found:", backbone.name if backbone else "None")

In [ ]:
# backbone ke andar last conv layer dhoondho
def find_last_conv_layer_name(backbone_model):
    for layer in reversed(backbone_model.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer.name
        except:
            pass
    return None

last_conv_layer_name = find_last_conv_layer_name(backbone)
print("Last conv layer:", last_conv_layer_name)

In [ ]:
def make_gradcam_heatmap_nested(img_array, model, backbone, last_conv_layer_name, pred_index=None):
    """
    Works for nested models like:
    Input -> Augmentation/Preprocess -> EfficientNetB0 -> GAP -> Dense
    """

    # input ko tensor banao
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)

    # model ko ek baar call kar do taaki graph ready ho jaye
    _ = model(img_tensor, training=False)

    # backbone ke chosen conv layer ka output
    last_conv_layer = backbone.get_layer(last_conv_layer_name)

    # submodel 1: backbone input -> last conv output
    last_conv_model = tf.keras.Model(
        inputs=backbone.input,
        outputs=last_conv_layer.output
    )

    # submodel 2: backbone output ke baad jo layers hain unko manually apply karenge
    classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])
    x = classifier_input

    passed_backbone = False
    for layer in model.layers:
        if layer.name == backbone.name:
            passed_backbone = True
            continue
        if passed_backbone:
            x = layer(x)

    classifier_model = tf.keras.Model(classifier_input, x)

    with tf.GradientTape() as tape:
        # outer model ke input ko backbone input format me preprocess karo
        x = img_tensor

        # model ke starting layers ko backbone tak apply karo
        for layer in model.layers:
            if layer.name == backbone.name:
                break
            x = layer(x, training=False)

        last_conv_layer_output = last_conv_model(x, training=False)
        tape.watch(last_conv_layer_output)

        preds = classifier_model(last_conv_layer_output, training=False)

        if pred_index is None:
            pred_index = tf.argmax(preds[0])

        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]

    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    denominator = tf.reduce_max(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (denominator + 1e-8)

    return heatmap.numpy(), preds.numpy()

In [ ]:
# test image select karo
nsfw_test_files = list((TEST_DIR / "nsfw").glob("*"))
safe_test_files = list((TEST_DIR / "safe").glob("*"))

if len(nsfw_test_files) > 0:
    img_path = str(nsfw_test_files[0])
else:
    img_path = str(safe_test_files[0])

print("Using image:", img_path)

img_array = get_img_array(img_path, IMAGE_SIZE)

heatmap, preds = make_gradcam_heatmap_nested(
    img_array=img_array,
    model=model,
    backbone=backbone,
    last_conv_layer_name=last_conv_layer_name
)

print("Prediction score:", preds[0][0])
print("✅ Heatmap generated")

In [ ]:
heatmap = make_gradcam_heatmap(
    img_array,
    model,
    base_model_name="efficientnetb0",
    last_conv_layer_name="top_conv"
)

In [ ]:
img_path = str(TEST_DIR / "nsfw" / os.listdir(TEST_DIR / "nsfw")[0])
last_conv_layer_name = "top_conv"

img_array = get_img_array(img_path, IMAGE_SIZE)
img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)

heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
print("✅ Heatmap created")